# Debug Pipeline RAG Pháp lý — chạy trên Kaggle (KHÔNG dùng ground truth)

Notebook chạy `debug_pipeline.py` trên bộ câu hỏi upload qua Kaggle Dataset, **log đầy đủ từng giai đoạn**
(Retrieval → Rerank → Prompt → LLM → Verify), rồi **xuất `results.json` đúng format nộp** để bạn zip nộp BTC.

Không so sánh ground truth (GT cũ của nhóm khác bị sai).

In [ ]:
# 1) Cài dependencies
!pip install -q qdrant-client rank_bm25 underthesea sentence-transformers \
    transformers accelerate bitsandbytes python-dotenv tqdm

In [ ]:
# 2) Load secrets Qdrant từ Kaggle
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    os.environ["QDRANT_URL"] = secrets.get_secret("QDRANT_URL")
    os.environ["QDRANT_API_KEY"] = secrets.get_secret("QDRANT_API_KEY")
    print("✅ Secrets loaded from Kaggle")
except Exception as e:
    print("⚠️ Không load được secrets:", e)

In [ ]:
# 3) Clone / pull code mới nhất từ GitHub
import os, sys

GITHUB_USERNAME = "kimmttrung"
REPO_NAME = "legal-rag-vietnam"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
WORKING_DIR = f"/kaggle/working/{REPO_NAME}"

if os.path.exists(WORKING_DIR):
    print("🔄 Pull code mới nhất...")
    !cd {WORKING_DIR} && git pull origin main
else:
    print("📥 Clone lần đầu...")
    !cd /kaggle/working && git clone {REPO_URL}

if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)
print("✅ Đồng bộ hoàn tất!")

In [ ]:
# 4) Vào thư mục repo + kiểm tra file cần thiết
import os, sys

WORKING_DIR = "/kaggle/working/legal-rag-vietnam"
os.chdir(WORKING_DIR)
if sys.path[0] != WORKING_DIR:
    sys.path.insert(0, WORKING_DIR)

print("Working directory:", os.getcwd())
for f in ["debug_pipeline.py", "config/settings.py", "data/corpus_clean.json", "data/law_manifest.json"]:
    print(("✅ " if os.path.exists(f) else "❌ THIẾU: ") + f)

In [ ]:
# 5) Tự tìm file câu hỏi trong Kaggle Dataset (/kaggle/input)
import glob, json

# Nếu auto-detect sai, gán thẳng đường dẫn vào INPUT_FILE bên dưới.
INPUT_FILE = None

if INPUT_FILE is None:
    candidates = glob.glob("/kaggle/input/**/*.json", recursive=True)
    print("Các file JSON tìm thấy trong /kaggle/input:")
    for c in candidates:
        print("  -", c)
    pick = next((c for c in candidates if any(k in c.lower() for k in ["r2ai", "question", "data", "test"])), None)
    INPUT_FILE = pick or (candidates[0] if candidates else None)

assert INPUT_FILE and os.path.exists(INPUT_FILE), "Không tìm thấy file input! Gán thủ công INPUT_FILE."

_raw = json.load(open(INPUT_FILE, encoding="utf-8"))
_qs = _raw if isinstance(_raw, list) else _raw.get("data", _raw.get("questions", []))
NUM_QUESTIONS = len(_qs)
print(f"\n✅ INPUT_FILE = {INPUT_FILE}  ({NUM_QUESTIONS} câu hỏi)")
print("Ví dụ câu đầu:", json.dumps(_qs[0], ensure_ascii=False)[:200] if _qs else "(rỗng)")

In [ ]:
# 6) Chạy debug pipeline — KHÔNG dùng ground truth (--ground-truth "")
#    Log từng giai đoạn lưu vào debug_output/
print(f"🚀 Chạy debug pipeline {NUM_QUESTIONS} câu...\n")
!python debug_pipeline.py \
    --input "{INPUT_FILE}" \
    --ground-truth "" \
    --num-questions {NUM_QUESTIONS} \
    --output-dir debug_output

## Xuất file nộp `results.json`

Lấy 5 trường chuẩn (`id, question, answer, relevant_docs, relevant_articles`) từ kết quả pipeline,
đảm bảo **đủ số bản ghi = số câu input** (câu lỗi vẫn có placeholder, không sót id).

In [ ]:
# 7) XUẤT results.json (FORMAT NỘP)
import json, zipfile

def _norm_id(v):
    # Giữ id dạng SỐ NGUYÊN nếu có thể ("1" -> 1), ngược lại giữ nguyên giá trị gốc
    if isinstance(v, bool):
        return v
    if isinstance(v, int):
        return v
    s = str(v).strip()
    return int(s) if s.lstrip("-").isdigit() else s

qs_raw = json.load(open(INPUT_FILE, encoding="utf-8"))
qs_all = qs_raw if isinstance(qs_raw, list) else qs_raw.get("data", qs_raw.get("questions", []))

final_map = {str(r["id"]): r for r in json.load(open("debug_output/05_final_result.json", encoding="utf-8"))}

results = []
for q in qs_all:
    orig_id = q.get("id", q.get("question_id", ""))
    out_id = _norm_id(orig_id)          # -> 1 (int) thay vì "1" (str)
    r = final_map.get(str(orig_id))
    if r:
        results.append({
            "id": out_id,
            "question": r.get("question", q.get("question", "")),
            "answer": r.get("answer", ""),
            "relevant_docs": r.get("relevant_docs", []),
            "relevant_articles": r.get("relevant_articles", []),
        })
    else:
        results.append({
            "id": out_id,
            "question": q.get("question", ""),
            "answer": "Không thể xử lý câu hỏi này do lỗi hệ thống.",
            "relevant_docs": [],
            "relevant_articles": [],
        })

out_json = "/kaggle/working/results.json"
json.dump(results, open(out_json, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"✅ {out_json} — {len(results)} bản ghi (input có {len(qs_all)} câu)")
print("   id mẫu:", [r["id"] for r in results[:5]], "| kiểu:", type(results[0]["id"]).__name__)

# Zip sẵn cho tiện (BTC nhận results.json nằm trong zip)
with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_json, arcname="results.json")
print("✅ /kaggle/working/submission.zip (chứa results.json)")

print("\n----- Bản ghi mẫu -----")
print(json.dumps(results[0], ensure_ascii=False, indent=2)[:700])

## Đọc log từng giai đoạn (chẩn đoán)

In [ ]:
# 8) Trace ĐẦY ĐỦ từng giai đoạn cho tất cả câu
import json

def load(name):
    return {str(r["id"]): r for r in json.load(open(f"debug_output/{name}", encoding="utf-8"))}

retr   = load("01_retrieval.json")
rerank = load("02_rerank.json")
gen    = load("03_llm_generation.json")
verif  = load("04_self_verification.json")
final  = load("05_final_result.json")

for qid in retr.keys():
    r = retr[qid]
    print("="*90)
    print(f"Q{qid}: {r['question']}")
    print("="*90)

    print(f"\n--- [1] RETRIEVAL --- dense={r['dense_count']} sparse={r['sparse_count']} merged={r['merged_count']}")
    print("Query variants:", r["query_variants"])
    for c in r["candidates"][:10]:
        rrf = c.get("rrf_score")
        print(f"  [{c['doc_number']}] {c['article_id']} | rrf={rrf:.4f} | {str(c['title'])[:55]}")

    rk = rerank[qid]
    smin, smax = rk.get("score_min"), rk.get("score_max")
    rng = f"[{smin:.3f}, {smax:.3f}]" if smin is not None else "N/A"
    print(f"\n--- [2] RERANK --- score={rng} threshold={rk['threshold']} fallback={rk['fallback_used']} final={rk['final_count']}")
    for c in rk["final_candidates"]:
        print(f"  [{c['doc_number']}] {c['article_id']} | rerank={c['rerank_score']:.3f} | {str(c['title'])[:55]}")

    g = gen[qid]
    p = g["prompt"]
    print(f"\n--- [3] PROMPT --- tokens={p['prompt_token_count']} | context_cắt={p['context_was_truncated']} | số_chunk={p['num_context_docs']}")

    print(f"\n--- [4] LLM GENERATION --- regenerated={g['was_regenerated']}")
    for a in g["attempts"]:
        print(f"  Lần {a['attempt']} (temp={a['temperature']}) verify_pass={a['verify_passed']}")
        print(f"    Answer: {a['answer'][:500]}")
        if a["verify_violations"]:
            print(f"    Vi phạm: {a['verify_violations']}")

    v = verif[qid]
    print(f"\n--- [5] VERIFICATION --- passed={v['passed']} attempts={v['num_attempts']} violations={v['final_violations']}")

    fn = final[qid]
    print(f"\n--- [6] KẾT QUẢ CUỐI ---")
    print("  relevant_docs:")
    for d in fn["relevant_docs"]:
        print("    -", d)
    print("  relevant_articles:")
    for a in fn["relevant_articles"]:
        print("    -", a)
    print(f"\n  Answer cuối: {fn['answer'][:600]}")
    print()

In [ ]:
# 9) (Tùy chọn) In TOÀN BỘ prompt gửi vào LLM của 1 câu
import json
QID = list(retr.keys())[0]   # đổi sang id câu khác nếu muốn
p = gen[QID]["prompt"]
print(f"Q{QID} | tokens={p['prompt_token_count']} | context_cắt={p['context_was_truncated']}")
print("="*90, "\nPROMPT GỬI VÀO LLM:\n", "="*90)
print(p["formatted_prompt"])

In [ ]:
# 10) In log hệ thống + đóng gói debug_output để tải về phân tích
print("------ logs/debug_pipeline.log (50 dòng cuối) ------")
_log = open("logs/debug_pipeline.log", encoding="utf-8").read().splitlines()
print("\n".join(_log[-50:]))

import shutil
shutil.make_archive("/kaggle/working/debug_pipeline", "zip", "debug_output")
print("\n✅ /kaggle/working/debug_pipeline.zip (log chẩn đoán)")
print("✅ /kaggle/working/results.json + submission.zip (FILE NỘP)")